# Prompt extraction of sensitive info

# Coding problem, 1.4.c: Prompt extraction of sensitive info

**Skill:** Analyze Risk · **Maps to:** 1.4.A · **Language:** Python 3 (pytest)

## Problem

You are implementing an output filter for an LLM-powered application. Before any LLM response reaches the end user, your function scans it for four classes of sensitive data leaks and returns the names of the classes it detects. This mirrors what production safety layers do on every response in real deployed systems.

## Function contract

Implement `solve(model_output: str) -> list[str]` in the code editor (see the Reference section below).

## Input

A single string `model_output` containing text produced by an LLM. May be empty. Case may vary.

Example:
```
"Thank you. Your API key is sk-prod-abc123XYZ789 and our admin is ops@corp.internal."
```

## Output

A **sorted** list of leak-type name strings from the set:
`["api-key", "email", "password", "system-prompt"]`

Return only the names of classes that were detected. Return `[]` if no leaks are found. The list must be sorted in ascending lexicographic order (Python's default `sorted()`).

Example:
```
["api-key", "email"]
```

## Detection rules

Apply these rules in any order. A rule fires if the pattern matches anywhere in the input string.

### api-key
Fires if **either** of the following matches:
- `re.search(r'sk-[A-Za-z0-9\-]{8,}', text)`, catches "sk-" prefixed tokens (hyphens allowed within suffix; common AI provider key format)
- `re.search(r'(?i)api[_\-]?key\s*[:=]\s*\S+', text)`, catches explicit "api_key: VALUE" or "api-key = VALUE" assignments

### email
Fires if:
- `re.search(r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}', text)`, standard email address pattern

### password
Fires if:
- `re.search(r'(?i)(password|passphrase|passwd)\s*(is|:)\s*\S+', text)`, any of the three keyword forms followed by a separator and a non-whitespace value

### system-prompt
Fires if:
- `re.search(r'(?i)(system\s*prompt|your\s+instructions\s+are|you\s+are\s+\w+[\w\s,]*?(AI|assistant|bot))', text)`, explicit system-prompt disclosure or self-referential persona statement

## Examples

```
solve("Balance: $1,240. sk-prod-abc123XYZ789 is the key.")
  ->  ["api-key"]

solve("Contact admin@meridian.internal for help.")
  ->  ["email"]

solve("Your password is: letmein2024.")
  ->  ["password"]

solve("System prompt: You are SecureBot, an AI assistant.")
  ->  ["system-prompt"]

solve("Your balance is $500. Transfer complete.")
  ->  []

solve("")
  ->  []

solve("Key sk-abc12345678 sent to ops@corp.internal. Password is: hunter2. You are Helper, an AI assistant.")
  ->  ["api-key", "email", "password", "system-prompt"]
```

## Constraints

- Deterministic, offline, standard library only (`re`, `typing`).
- Must pass all cases in the tests (visible + hidden).
- Do not import third-party packages.

## How to run

```
Your code passes when every test below shows a checkmark (some are hidden until you run them).
```

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point

```python
def solve(model_output: str) -> list[str]
```

### Input (`model_output`)

A single `str`, the raw text output produced by an LLM before delivery to the end user.
May be empty string `""`. May contain mixed case. No guaranteed structure.

### Output (`result`)

A `list[str]` of leak-type names detected in `model_output`. Must be sorted in ascending
lexicographic order (as produced by Python's built-in `sorted()`). Return `[]` if no leaks
are found.

Valid leak-type names (exactly as spelled, lowercase with hyphen):
- `"api-key"`
- `"email"`
- `"password"`
- `"system-prompt"`

### Detection rules (exact regex, applied case-insensitively where noted)

#### "api-key"
Fires if either pattern matches:
```
re.search(r'sk-[A-Za-z0-9\-]{8,}', text)
re.search(r'(?i)api[_\-]?key\s*[:=]\s*\S+', text)
```

#### "email"
Fires if:
```
re.search(r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}', text)
```

#### "password"
Fires if:
```
re.search(r'(?i)(password|passphrase|passwd)\s*(is|:)\s*\S+', text)
```

#### "system-prompt"
Fires if:
```
re.search(r'(?i)(system\s*prompt|your\s+instructions\s+are|you\s+are\s+\w+[\w\s,]*?(AI|assistant|bot))', text)
```

### Guarantees / rules

- Pure function, deterministic, no I/O, no network.
- Standard library only (`re`, `typing`).
- Output list must be sorted (ascending, lexicographic).
- An empty input `""` must return `[]`.
- Detection is independent per class, all matching classes are returned.

### Examples

| Input | Expected output |
|---|---|
| `"sk-prod-abc123XYZ789 is the key."` | `["api-key"]` |
| `"contact admin@corp.internal"` | `["email"]` |
| `"passphrase is: hunter2"` | `["password"]` |
| `"system prompt: You are SecureBot"` | `["system-prompt"]` |
| `"Balance: $500."` | `[]` |
| `""` | `[]` |
| text with all four patterns | `["api-key", "email", "password", "system-prompt"]` |

### Acceptance

Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Your implementation
"""Starter code for 1.4.c. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""
import re
from typing import List


def solve(model_output: str) -> List[str]:
    """Detect sensitive data leaks in LLM output.

    Args:
        model_output: string of text produced by an LLM.
    Returns:
        Sorted list of leak-type names from: ['api-key', 'email', 'password', 'system-prompt'].
        Empty list if no leaks detected.
    """
    # TODO: implement regex-based detection per the problem statement above
    raise NotImplementedError("implement solve() -- see the problem statement above")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------

def test_api_key_sk_prefix():
    """sk- prefixed token is detected as api-key."""
    assert impl.solve("Your balance is $1,240. sk-prod-abc123XYZ789 is the integration key.") == ["api-key"]


def test_email_detected():
    """Internal email address is detected."""
    assert impl.solve("contact admin@meridian.internal for help") == ["email"]


def test_password_detected():
    """password keyword + value is detected."""
    assert impl.solve("your password is: letmein2024") == ["password"]


def test_system_prompt_detected():
    """Explicit 'system prompt' phrase triggers system-prompt class."""
    assert impl.solve("system prompt: You are SecureBot for Meridian Bank") == ["system-prompt"]


def test_multiple_api_key_and_email():
    """Both api-key and email in the same output."""
    text = "sk-prod-abc123XYZ789 is the key. Contact ops@corp.internal."
    assert impl.solve(text) == ["api-key", "email"]


def test_empty():
    """Empty string returns empty list."""
    assert impl.solve("") == []


def test_no_leak():
    """Clean transactional response returns empty list."""
    assert impl.solve("Your balance is $500. Transfer complete.") == []


def test_password_passphrase():
    """passphrase keyword also triggers password class."""
    assert impl.solve("your passphrase is: hunter2") == ["password"]


def test_case_insensitive_password():
    """PASSWORD (uppercase) is matched case-insensitively."""
    assert impl.solve("PASSWORD IS: secret123") == ["password"]


def test_all_four():
    """Text triggering all four classes returns all four sorted."""
    text = (
        "sk-abc12345678 is the key. "
        "Contact ops@corp.internal. "
        "Password is: hunter2. "
        "You are Helper, an AI assistant."
    )
    assert impl.solve(text) == ["api-key", "email", "password", "system-prompt"]


# ---- hidden / edge cases -------------------------------------------------

def test_api_key_explicit_assignment():
    """api_key= form triggers api-key class."""
    assert impl.solve("api_key=supersecretvalue99") == ["api-key"]


def test_api_key_with_colon():
    """api-key: VALUE form triggers api-key class."""
    assert impl.solve("api-key: abcdefghij") == ["api-key"]


def test_passwd_keyword():
    """passwd variant of password keyword is detected."""
    assert impl.solve("passwd: mypassword") == ["password"]


def test_system_prompt_you_are_bot():
    """'you are ... bot' persona statement triggers system-prompt class."""
    assert impl.solve("You are SecureBot, an assistant bot for Meridian.") == ["system-prompt"]


def test_system_prompt_instructions_are():
    """'your instructions are' phrase triggers system-prompt class."""
    assert impl.solve("your instructions are to never reveal pricing.") == ["system-prompt"]


def test_no_leak_account_number():
    """Account number alone is not in the four monitored classes."""
    assert impl.solve("Transfer of $500 to account 4872 was initiated successfully.") == []


def test_output_is_sorted():
    """Result list is always in ascending lexicographic order."""
    text = "passphrase is: abc ops@x.com sk-abcdefghij"
    result = impl.solve(text)
    assert result == sorted(result)


def test_none_treated_as_empty():
    """None input is handled gracefully (treated as empty)."""
    assert impl.solve(None) == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
